# Mission Control — Hugging Face, senza Ollama
GLiNER2.5-multi-Decide su CPU + Gemma 3 4B IT sulla GPU, entrambi tramite librerie Python.

1. Scegli un runtime GPU.
2. Accetta le condizioni del repository `google/gemma-3-4b-it` sul sito Hugging Face.
3. Esegui le celle in ordine e autentica lo stesso account con un token di lettura.
4. Carica il JSON delle route e il TXT degli esempi.

Nessun subprocess e nessun server esterno. La sola installazione usa `%pip`.
I pesi vengono scaricati nella VM di Colab. Le risorse GPU non sono garantite.
La demo produce piani, ma non esegue strumenti. Il quinto esempio non garantisce score basso.


In [ ]:
%pip install --upgrade --upgrade-strategy only-if-needed \
    "pandas==2.2.3" \
    "huggingface-hub>=0.25.0,<2.0" \
    "gliner2[local]" \
    transformers \
    accelerate \
    ipywidgets

Se Colab richiede un riavvio dopo l’installazione, riavvia la sessione e continua dalle importazioni.

In [ ]:
import json, re, time
from pathlib import Path
import torch
import pandas as pd
from IPython.display import display, clear_output, Markdown
from huggingface_hub import notebook_login

if not torch.cuda.is_available():
    raise RuntimeError("Attiva un runtime GPU prima di procedere.")
props = torch.cuda.get_device_properties(0)
print(props.name, f"{props.total_memory / 2**30:.1f} GiB")
notebook_login()


In [ ]:
from google.colab import files
uploaded = files.upload()
json_names = [name for name in uploaded if name.lower().endswith(".json")]
txt_names = [name for name in uploaded if name.lower().endswith(".txt")]
if len(json_names) != 1 or len(txt_names) != 1:
    raise ValueError("Carica esattamente un JSON delle route e un TXT degli esempi.")
routes = json.loads(uploaded[json_names[0]].decode("utf-8-sig"))["routes"]
raw_text = uploaded[txt_names[0]].decode("utf-8-sig")
texts = [s.strip() for s in re.split(r"=== TESTO \d+ ===", raw_text) if s.strip()]
route_ids = {r["id"] for r in routes}
if len(route_ids) != len(routes) or not routes or not texts:
    raise ValueError("Route duplicate, route vuote o nessun testo.")
print(f"Caricati {len(routes)} percorsi e {len(texts)} testi.")


In [ ]:
from gliner2 import AutoExtractor

router = AutoExtractor.from_pretrained("fastino/GLiNER2.5-multi-Decide")
router.to("cpu")
router.eval()
route_schema = router.create_schema().classification(
    "route", {r["id"]: r["description"] for r in routes}
)

def classify_request(text):
    result = router.extract(text, route_schema, include_confidence=True)
    prediction = result.get("route")
    if not isinstance(prediction, dict) or "confidence" not in prediction:
        raise RuntimeError(f"Formato confidence inatteso: {result}")
    route = prediction.get("label")
    score = float(prediction["confidence"])
    if route not in route_ids or not 0 <= score <= 1:
        raise ValueError(f"Predizione non valida: {prediction}")
    return route, score

measurements = []
for i, text in enumerate(texts, 1):
    route, score = classify_request(text)
    measurements.append({"testo": i, "route": route, "score": score})
display(pd.DataFrame(measurements))


## Caricamento di Gemma sulla GPU
Si carica Gemma in FP16, non il tag quantizzato di Ollama. Le risposte possono differire.
Non viene usato `device_map="auto"`: il modello deve stare sulla GPU; un errore di memoria è esplicito.
In caso di memoria insufficiente usa un runtime con più VRAM o adatta il caricamento alla quantizzazione.
GLiNER rimane sulla CPU.


In [ ]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

GEMMA_ID = "google/gemma-3-4b-it"
processor = AutoProcessor.from_pretrained(GEMMA_ID)
planner = Gemma3ForConditionalGeneration.from_pretrained(
    GEMMA_ID,
    torch_dtype=torch.float16,
    device_map={"": 0},
).eval()
print(f"Memoria GPU allocata: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")


In [ ]:
SYSTEM_PROMPT = """Sei un planner. Produci soltanto JSON valido, senza Markdown.
Non eseguire strumenti, non inventare policy, cambi o risultati. Non prenotare.
Gli strumenti disponibili sono quelli forniti qui sotto.
Usa riferimenti simbolici ai risultati dei passaggi precedenti.
Puoi combinare strumenti e porre domande di chiarimento.
Non chiedere subito dati recuperabili dagli strumenti.
Se i documenti risultassero incompleti, pianifica un ramo condizionale di chiarimento.
Formato:
{"steps":[{"id":"step_1","tool":"ID_STRUMENTO","task":"compito",
"depends_on":[],"run_if":"always oppure condizione esplicita"}],
"completion_condition":"condizione per fornire il risultato finale"}
Le dipendenze devono riferirsi a passaggi precedenti.
Strumenti disponibili:
""" + json.dumps(routes, ensure_ascii=False)

def validate_plan(plan):
    if not isinstance(plan, dict) or not isinstance(plan.get("steps"), list) or not plan["steps"]:
        raise ValueError("Il piano non contiene passaggi validi.")
    seen = set()
    for step in plan["steps"]:
        if not isinstance(step, dict):
            raise ValueError("Passaggio non valido.")
        sid = step.get("id")
        if not isinstance(sid, str) or not sid or sid in seen:
            raise ValueError("ID mancante o duplicato.")
        if step.get("tool") not in route_ids:
            raise ValueError("Strumento non consentito.")
        deps = step.get("depends_on")
        if not isinstance(deps, list) or any(not isinstance(d, str) or d not in seen for d in deps):
            raise ValueError("Dipendenze mancanti, future o cicliche.")
        for field in ("task", "run_if"):
            if not isinstance(step.get(field), str) or not step[field].strip():
                raise ValueError(f"Campo mancante: {field}")
        seen.add(sid)
    if not isinstance(plan.get("completion_condition"), str):
        raise ValueError("completion_condition mancante.")
    return plan

def generate_plan(text):
    # Un unico messaggio utente evita dipendenze dal supporto del ruolo system.
    messages = [{"role":"user", "content":[{"type":"text", "text":
        SYSTEM_PROMPT + "\n\nRICHIESTA UTENTE:\n" + text}]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to("cuda:0")
    prompt_length = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output = planner.generate(**inputs, max_new_tokens=1600, do_sample=False)
    raw = processor.decode(output[0, prompt_length:], skip_special_tokens=True).strip()
    # Accetta soltanto un eventuale fence esterno, non estrae JSON arbitrario dal testo.
    cleaned = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned).strip()
    try:
        plan = validate_plan(json.loads(cleaned))
        return {"valid": True, "plan": plan, "raw_output": raw, "error": None}
    except (ValueError, TypeError, KeyError) as exc:
        return {"valid": False, "plan": None, "raw_output": raw, "error": str(exc)}

def run_demo(text, threshold=0.8, force_llm=False):
    start = time.perf_counter()
    route, score = classify_request(text)
    router_ms = (time.perf_counter() - start) * 1000
    fallback = force_llm or score < threshold
    if fallback:
        torch.cuda.synchronize()
        llm_start = time.perf_counter()
        generation = generate_plan(text)
        torch.cuda.synchronize()
        planner_ms = (time.perf_counter() - llm_start) * 1000
    else:
        generation = {"valid": True, "plan": {
            "steps":[{"id":"step_1", "tool":route, "task":text,
                      "depends_on":[], "run_if":"always"}],
            "completion_condition":"Eseguire il percorso selezionato; strumenti non eseguiti nella demo."
        }, "raw_output":None, "error":None}
        planner_ms = 0
    return {"route":route, "score":score, "threshold":threshold,
            "mode":"llm_planner" if fallback else "direct",
            "fallback_reason":"manual_override" if force_llm else ("below_threshold" if fallback else None),
            "router_ms":round(router_ms, 1), "planner_ms":round(planner_ms, 1),
            "total_ms":round((time.perf_counter()-start)*1000, 1), **generation}


## Interfaccia
La soglia è dimostrativa e lo score non è una probabilità calibrata di correttezza.
Il fallback può non attivarsi sul caso composto: un modello può sbagliare con score alto.
L'override manuale è segnalato esplicitamente. Il JSON viene validato strutturalmente, non semanticamente.
La prima generazione può includere costi di inizializzazione; i tempi non sono un benchmark.


In [ ]:
import ipywidgets as widgets

selector = widgets.Dropdown(options=[(f"Testo {i}", t) for i,t in enumerate(texts,1)], description="Esempio:")
text_box = widgets.Textarea(value=texts[0], layout=widgets.Layout(width="100%", height="160px"))
selector.observe(lambda c: setattr(text_box, "value", c["new"]), names="value")
threshold = widgets.FloatSlider(value=0.8, min=0, max=1, step=0.01, description="Soglia:")
force = widgets.Checkbox(value=False, description="Forza LLM (override manuale)")
button = widgets.Button(description="Esegui", button_style="primary")
output_area = widgets.Output()

def on_run(_):
    button.disabled = True
    with output_area:
        clear_output(wait=True)
        try:
            if not text_box.value.strip():
                raise ValueError("Inserisci una richiesta.")
            result = run_demo(text_box.value, threshold.value, force.value)
            branch = "Gemma GPU → Piano" if result["mode"] == "llm_planner" else f"Route diretta: {result['route']}"
            display(Markdown(f"### Input → GLiNER CPU → Gate → {branch}"))
            if not result["valid"]:
                display(Markdown("### Piano non valido: controllare errore e output grezzo"))
            print(json.dumps(result, indent=2, ensure_ascii=False))
        except Exception as exc:
            print(f"{type(exc).__name__}: {exc}")
    button.disabled = False

button.on_click(on_run)
display(selector, text_box, threshold, force, button, output_area)
